# Trazo en Google Colab

Servidor del modelo propio **trazo** (chat, Qwen 2.5 Coder 7B + Modelfile de UML Studio) y de **qwen3-vl:8b-instruct** (lectura de imágenes) con Ollama en una GPU de Colab, publicado con un túnel de Cloudflare.

1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*.
2. Ejecuta las celdas en orden.
3. Deja ejecutándose la última celda (mantiene viva la sesión).
4. Copia la URL `https://….trycloudflare.com` que imprime la última celda en el `.env` del backend: `OLLAMA_BASE_URL=<url>` y reinicia el backend.

Mientras el cuaderno siga abierto, el backend usa este servidor. Cualquiera con la URL puede usar el modelo: no la publiques.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Instalar Ollama
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Arrancar el servidor en segundo plano
import os, subprocess, time
env = dict(os.environ, OLLAMA_HOST='0.0.0.0:11434', OLLAMA_KEEP_ALIVE='24h', OLLAMA_MAX_LOADED_MODELS='2')
ollama = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
!curl -s http://localhost:11434/api/version

In [ ]:
%%writefile Modelfile
# Mismo contenido que ollama/Modelfile del backend
FROM qwen2.5-coder:7b
PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.05
PARAMETER num_ctx 8192
SYSTEM """
Eres Trazo, desarrollador senior y mentor de modelado UML 2.5 de la herramienta UML Studio.
Acompañas a un equipo que diseña diagramas de clases en una pizarra compartida y que después
genera un backend Spring Boot (JPA + PostgreSQL) a partir del diagrama. Hablas en español,
llamas a cada persona por su nombre, explicas el porqué de cada decisión de modelado y
propones mejoras concretas sin inventar elementos que no existan en el diagrama.
"""

In [ ]:
# Descargar el modelo base y crear el modelo propio
!ollama pull qwen2.5-coder:7b
!ollama create trazo -f Modelfile
!ollama run trazo "Explica en dos frases qué es un atributo derivado en UML."
!ollama pull qwen3-vl:8b-instruct
!ollama list


In [ ]:
# Precargar ambos modelos en la GPU para que la primera petición del backend no espere
!curl -s http://localhost:11434/api/generate -d '{"model":"trazo","prompt":"hola","stream":false,"keep_alive":"24h"}' > /dev/null
!curl -s http://localhost:11434/api/generate -d '{"model":"qwen3-vl:8b-instruct","prompt":"hola","stream":false,"think":false,"keep_alive":"24h"}' > /dev/null
!ollama ps

In [ ]:
# Publicar el servidor con un túnel de Cloudflare (sin cuenta)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import re
tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--http-host-header', 'localhost:11434'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        print('OLLAMA_BASE_URL=' + match.group(0))
        break

In [ ]:
# Mantener viva la sesión: deja esta celda EJECUTÁNDOSE mientras uses la app.
import time
while True:
    time.sleep(60)
    tail = open('ollama.log', errors='replace').read().strip().splitlines()[-1:]
    print(time.strftime('%H:%M'), tail[0][:150] if tail else '')
